# 第 8 章　CFD Trading with Oanda

> *Today, even small entities that trade complex instruments or are granted sufficient leverage can threaten the global financial system.* —— Paul Singer

## 0. 引言

选择交易平台要考虑什么?

如今入门金融市场交易比以往任何时候都容易。可供算法交易者选择的在线交易平台很多，选择时可能受以下因素影响：

- Instruments（交易品种）：股票、ETF、债券、货币、大宗商品、期权、期货等。
- Strategies（策略）：有的交易者只做多，有的还需要做空；有的只做单一品种，有的需要同时交易多个品种。
- Costs（成本）：固定与可变交易成本，甚至可能决定某个策略是否盈利。
- Technology（技术）：一是平台提供的交易工具（桌面/笔记本、平板、手机）；二是可通过程序访问的 API。
- Jurisdiction（司法辖区）：金融交易监管严格，各国/地区法律框架不同，可能限制某些交易者使用某些平台和品种。

本章聚焦 Oanda —— 一个适合部署自动化算法交易策略的在线平台，零售交易者也能使用。按上述维度简述如下：

- Instruments：提供大量差价合约（CFD）。 CFD 的主要特征是带杠杆（如 10:1 或 50:1）、保证金交易，因此亏损可能超过初始本金。
- Strategies：可做多也可做空；支持市价单、限价单，可附带止盈和（追踪）止损。
- Costs：没有固定交易费用，但有买卖价差（bid-ask spread），形成可变交易成本。
- Technology：提供 fxTrade (Practice) 桌面应用（实时行情、手动交易）和浏览器版应用。最大优势是 RESTful 与流式 API（Oanda v20 API），可编程获取历史/实时数据、下单、查询账户信息；有 Python 封装包（PyPI 上的 `v20`）。 Oanda 提供免费模拟账户，拥有完整技术能力，便于入门以及从模拟过渡到实盘。
- Jurisdiction：可交易的 CFD 取决于账户持有人的居住地。外汇类 CFD 基本在 Oanda 开展业务的所有地区都可交易；股指 CFD 等在某些地区可能不可用。

### 0.1 差价合约（Contracts for Difference, CFDs）

CFD 的更多细节可参考 Investopedia 与 Wikipedia 的 CFD 词条。CFD 可基于：

- 货币对（如 EUR/USD），
- 大宗商品（如黄金）、
- 股指（如标普 500）、
- 债券（如德国 10 年期 Bund）
  
等，产品覆盖面足以实现全球宏观策略。

从金融角度看，CFD 是衍生品，其收益取决于其他标的价格的变化；此外，交易活跃度也会影响 CFD 价格。即使某个 CFD 以标普 500 指数为标的，它仍然是由 Oanda（或类似机构）自行发行、报价和支持的另一种产品。这带来一定风险。近期的例子是 瑞士法郎事件（SNB 取消汇率下限），导致多家在线经纪商破产。

本 notebook 全章代码使用 Python 封装类 `tpqoa`（GitHub 仓库 `yhilpisch/tpqoa`）。 目标是让用户学会运用前面章节的方法与技术，在 Oanda 平台上自动交易。

## 1. Setting Up an Account

在 Oanda 开户简单高效。可选真实账户或免费模拟（practice）账户，后者完全足以完成本章内容。

注册成功并登录后会看到起始页，中间有 fxTrade Practice for Desktop 的下载链接，安装。

## 2. The Oanda API

注册后访问 Oanda API 很容易，需要两样东西：账户号（account number）和访问令牌（access token，即 API key）。

- 账户号：可在 Manage Funds 区域查看
- 访问令牌：可在 Manage API Access 区域生成

⚠️ 注意：Oanda API 中的命名并不完全一致 -- API key 与 access token 可互换使用；account ID 与 account number 指同一个号码。

此后用 `configparser` 模块管理账户凭证。与天勤笔记的 `tqsdk.cfg` 一样，在笔记本同目录创建 `oanda.cfg`。使用 `configparser` 的好处是便于存储和管理账户凭证。算法交易中需要的账户数量会迅速增加（云实例/服务器、数据服务商、在线交易平台等）。缺点是账户信息以纯文本存储，存在相当大的安全风险，尤其是多个账户信息放在同一个文件里时。进入生产环境时，应对文件加密以保护凭证。模拟账户格式如下：

```ini
[oanda]
account_id = YOUR_ACCOUNT_ID
access_token = YOUR_ACCESS_TOKEN
account_type = practice
```

⚠️ 不论是`tqsdk.cfg` 还是 `oanda.cfg`，请勿提交至 Git！

通过 Python 访问 API，推荐使用封装包 `tpqoa`，它又依赖 Oanda 官方的 `v20` 包。安装命令：


In [13]:
# !pip install -U setuptools wheel -i https://mirrors.aliyun.com/pypi/simple
# !pip install git+https://github.com/yhilpisch/tpqoa.git -i https://mirrors.aliyun.com/pypi/simple

满足上述前提后，一行代码即可连接 API（路径与文件名请按需调整）：

In [10]:
import tpqoa

In [11]:
api = tpqoa.tpqoa('oanda.cfg')   # ① 读取含凭证的配置文件

这是一个重要的里程碑：连上 Oanda API 之后，就可以获取历史数据、以编程方式下单等。

## 3. Retrieving Historical Data

Oanda 平台的一大好处：所有 Oanda 品种的完整价格历史都可通过 RESTful API 获取。这里的"完整历史"指的是各个 CFD 本身的历史，而非其标的资产的历史。

### 3.1 查询可交易品种

用 `.get_instruments()` 方法查看某账户可交易的品种。它只返回显示名称和技术名称；API 还能提供更多细节，例如最小持仓量：


In [ ]:
# 国内无法连接 Oanda，以下为书中 Out[3]（api.get_instruments()[:15]）
instruments = [
    ('AUD/CAD', 'AUD_CAD'),
    ('AUD/CHF', 'AUD_CHF'),
    ('AUD/HKD', 'AUD_HKD'),
    ('AUD/JPY', 'AUD_JPY'),
    ('AUD/NZD', 'AUD_NZD'),
    ('AUD/SGD', 'AUD_SGD'),
    ('AUD/USD', 'AUD_USD'),
    ('Australia 200', 'AU200_AUD'),
    ('Brent Crude Oil', 'BCO_USD'),
    ('Bund', 'DE10YB_EUR'),
    ('CAD/CHF', 'CAD_CHF'),
    ('CAD/HKD', 'CAD_HKD'),
    ('CAD/JPY', 'CAD_JPY'),
    ('CAD/SGD', 'CAD_SGD'),
    ('CHF/HKD', 'CHF_HKD'),
]
instruments


[
 ('AUD/CAD', 'AUD_CAD'),
 ('AUD/CHF', 'AUD_CHF'),
 ('AUD/HKD', 'AUD_HKD'),
 ('AUD/JPY', 'AUD_JPY'),
 ('AUD/NZD', 'AUD_NZD'),
 ('AUD/SGD', 'AUD_SGD'),
 ('AUD/USD', 'AUD_USD'),
 ('Australia 200', 'AU200_AUD'),
 ('Brent Crude Oil', 'BCO_USD'),
 ('Bund', 'DE10YB_EUR'),
 ('CAD/CHF', 'CAD_CHF'),
 ('CAD/HKD', 'CAD_HKD'),
 ('CAD/JPY', 'CAD_JPY'),
 ('CAD/SGD', 'CAD_SGD'),
 ('CHF/HKD', 'CHF_HKD')
]

### 3.2 在分钟线上回测动量策略

下例使用基于 EUR/USD 货币对的 `EUR_USD` 品种，目标是在1 分钟 K 线上回测基于动量的策略，数据为两天（见下方参数）。

第一步：从 Oanda 获取原始数据。先查看 `.get_history()` 的帮助文档：


In [5]:
help(api.get_history)

Help on method get_history in module tpqoa.tpqoa:

get_history(instrument, start, end, granularity, price, localize=True)
    method of tpqoa.tpqoa.tpqoa instance
    Retrieves historical data for instrument.

    Parameters
    instrument: string
        valid instrument name
    start, end: datetime, str
        Python datetime or string objects for start and end
    granularity: string
        a string like 'S5', 'M1' or 'D'
    price: string
        one of 'A' (ask), 'B' (bid) or 'M' (middle)

    Returns
    data: pd.DataFrame
        pandas DataFrame object with data


In [6]:
instrument = 'EUR_USD'
start = '2020-08-10'
end = '2020-08-12'
granularity = 'M1'   # ② 1 分钟 K 线
price = 'M'          # ③ 中间价（mid）

In [7]:
data = api.get_history(instrument, start, end,
                       granularity, price)   # ④ 从 API 取回原始数据

In [8]:
data.info()   # ⑤ 查看数据集的元信息

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2814 entries, 2020-08-10 00:00:00 to 2020-08-11 23:59:00
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   o         2814 non-null   float64
 1   h         2814 non-null   float64
 2   l         2814 non-null   float64
 3   c         2814 non-null   float64
 4   volume    2814 non-null   int64
 5   complete  2814 non-null   bool
dtypes: bool(1), float64(4), int64(1)
memory usage: 134.7 KB


In [9]:
data[['c', 'volume']].head()   # ⑥ 查看其中两列的前五行

                           c  volume
time
2020-08-10 00:00:00  1.17822      18
2020-08-10 00:01:00  1.17836      32
2020-08-10 00:02:00  1.17828      25
2020-08-10 00:03:00  1.17834      13
2020-08-10 00:04:00  1.17847      43

第二步：实现向量化回测。思路是同时回测几个动量策略（参见第 4 章）。为简单起见，下面只使用中间价的收盘价（`c`）。

In [10]:
import numpy as np

In [11]:
data['returns'] = np.log(data['c'] / data['c'].shift(1))   # ① 基于中间价收盘价计算对数收益率

In [12]:
cols = []   # ② 空列表，用来收集列名

In [13]:
for momentum in [15, 30, 60, 120]:   # ③ 动量窗口（分钟 K 线个数）
    col = 'position_{}'.format(momentum)   # ④ 存放仓位的列名
    data[col] = np.sign(data['returns'].rolling(momentum).mean())   # ⑤ 新增一列：策略仓位（过去 N 根收益均值的符号）
    cols.append(col)   # ⑥ 把列名追加进列表

最后一步：计算并绘制各动量策略的绝对表现。图 8-7 将各策略的表现与基础品种本身对比。

In [14]:
from pylab import plt
plt.style.use('seaborn')
import matplotlib as mpl
mpl.rcParams['savefig.dpi'] = 300
mpl.rcParams['font.family'] = 'serif'

In [15]:
strats = ['returns']   # ① 另建列表，保存稍后要画图的列名

In [16]:
for col in cols:   # ② 遍历各策略的仓位列
    strat = 'strategy_{}'.format(col.split('_')[1])   # ③ 生成存放策略表现的新列名
    data[strat] = data[col].shift(1) * data['returns']   # ④ 计算策略对数收益并存为新列（仓位滞后一期，避免前视偏差）
    strats.append(strat)   # ⑤ 把列名追加进列表，供后面画图

In [17]:
data[strats].dropna().cumsum(
        ).apply(np.exp).plot(figsize=(10, 6));   # ⑥ 绘制品种与各策略的累计表现

### 3.3 考虑杠杆与保证金

买入一只 100 美元的股票时，盈亏计算很直接：涨 1 美元赚 1 美元，跌 1 美元亏 1 美元；买 10 股则乘以 10。

在 Oanda 上交易 CFD 则涉及杠杆（leverage）和保证金（margin），这会显著影响盈亏计算。更多信息见 *Oanda fxTrade Margin Rules*。下面用一个简单例子说明要点。

例：一位以欧元计价的算法交易者想交易 `EUR_USD`，希望以 1.1 的卖价获得 10,000 EUR 的多头敞口。

- 不用杠杆：买入 10,000 个单位的 CFD。价格升至 1.105（买卖价中点）时，绝对盈利 = 10,000 × 0.005 = 50，即 0.5%。
- 20:1 杠杆（保证金 5% = 100%/20）：只需预付保证金 10,000 EUR × 5% = 500 EUR。价格升至 1.105，绝对盈利仍是 50 EUR，但相对收益为 50 / 500 = 10%——收益被放大 20 倍。这是事情顺利时杠杆的好处。
- 如果事情不顺：价格跌至 1.08，亏损 = 10,000 × (1.08 − 1.1) = −200 EUR，相对亏损为 −200 / 500 = −40%。如果账户中的权益/现金不足 200 EUR，由于（监管）保证金要求无法满足，仓位必须被平掉；若亏损耗尽全部保证金，就需要追加资金作为保证金才能维持交易。

In [18]:
data[strats].dropna().cumsum().apply(
    lambda x: x * 20).apply(np.exp).plot(figsize=(10, 6));   # ① 按假设的杠杆比例，把对数收益乘以 20

## 4. Working with Streaming Data（流式数据）

借助 `tpqoa` 与 `v20` 包，流式数据的处理同样简单：它们负责 socket 通信，算法交易者只需决定如何处理流进来的数据。


In [19]:
instrument = 'EUR_USD'

In [20]:
api.stream_data(instrument, stop=10)   # ① stop 参数：取回指定数量的 tick 之后停止流

2020-08-19T14:39:13.560138152Z 1.19131 1.1915
2020-08-19T14:39:14.088511060Z 1.19134 1.19152
2020-08-19T14:39:14.390081879Z 1.19124 1.19145
2020-08-19T14:39:15.105974700Z 1.19129 1.19144
2020-08-19T14:39:15.375370451Z 1.19128 1.19144
2020-08-19T14:39:15.501380756Z 1.1912 1.19141
2020-08-19T14:39:15.951793928Z 1.1912 1.19138
2020-08-19T14:39:16.354844135Z 1.19123 1.19138
2020-08-19T14:39:16.661440356Z 1.19118 1.19133
2020-08-19T14:39:16.912150908Z 1.19112 1.19132


## 5. Placing Market Orders（下市价单）

同样，用 `create_order()` 方法可以很方便地下市价买单/卖单。先看帮助文档：

In [21]:
help(api.create_order)

Help on method create_order in module tpqoa.tpqoa:

create_order(instrument, units, price=None, sl_distance=None,
tsl_distance=None, tp_price=None, comment=None, touch=False,
suppress=False, ret=False) method of tpqoa.tpqoa.tpqoa instance
    Places order with Oanda.

    Parameters
    instrument: string
        valid instrument name
    units: int
        number of units of instrument to be bought
        (positive int, e.g., 'units=50')
        or to be sold (negative int, e.g., 'units=-100')
    price: float
        limit order price, touch order price
    sl_distance: float
        stop loss distance price, mandatory e.g., in Germany
    tsl_distance: float
        trailing stop loss distance
    tp_price: float
        take profit price to be used for the trade
    comment: str
        string
    touch: boolean
        market_if_touched order (requires price to be set)
    suppress: boolean
        whether to suppress print out
    ret: boolean
        whether to return the order

上面的帮助文档展示了下市价单、限价单、触价市价单（market-if-touched）的全部选项。下面依次演示三笔市价单：


In [22]:
api.create_order(instrument, 1000)   # ① 市价单开多头 1000 单位

{'id': '1721', 'time': '2020-08-19T14:39:17.062399275Z', 'userID':
13834683, 'accountID': '101-004-13834683-001', 'batchID': '1720',
'requestID': '24716258589170956', 'type': 'ORDER_FILL', 'orderID':
'1720', 'instrument': 'EUR_USD', 'units': '1000.0',
'gainQuoteHomeConversionFactor': '0.835288642787',
'lossQuoteHomeConversionFactor': '0.843683503518', 'price': 1.19131,
'fullVWAP': 1.19131, 'fullPrice': {'type': 'PRICE', 'bids': [{'price':
1.1911, 'liquidity': '10000000'}], 'asks': [{'price': 1.19131,
'liquidity': '10000000'}], 'closeoutBid': 1.1911, 'closeoutAsk':
1.19131}, 'reason': 'MARKET_ORDER', 'pl': '0.0', 'financing': '0.0',
'commission': '0.0', 'guaranteedExecutionFee': '0.0',
'accountBalance': '98510.7986', 'tradeOpened': {'tradeID': '1721',
'units': '1000.0', 'price': 1.19131, 'guaranteedExecutionFee': '0.0',
'halfSpreadCost': '0.0881', 'initialMarginRequired': '33.3'},
'halfSpreadCost': '0.0881'}

In [23]:
api.create_order(instrument, -1500)   # ② 平掉多头后转为空头：卖出 1500（先平 1000，再开 500 空）

{'id': '1723', 'time': '2020-08-19T14:39:17.200434462Z', 'userID':
13834683, 'accountID': '101-004-13834683-001', 'batchID': '1722',
'requestID': '24716258589171315', 'type': 'ORDER_FILL', 'orderID':
'1722', 'instrument': 'EUR_USD', 'units': '-1500.0',
'gainQuoteHomeConversionFactor': '0.835288642787',
'lossQuoteHomeConversionFactor': '0.843683503518', 'price': 1.1911,
'fullVWAP': 1.1911, 'fullPrice': {'type': 'PRICE', 'bids': [{'price':
1.1911, 'liquidity': '10000000'}], 'asks': [{'price': 1.19131,
'liquidity': '9999000'}], 'closeoutBid': 1.1911, 'closeoutAsk':
1.19131}, 'reason': 'MARKET_ORDER', 'pl': '-0.1772', 'financing':
'0.0', 'commission': '0.0', 'guaranteedExecutionFee': '0.0',
'accountBalance': '98510.6214', 'tradeOpened': {'tradeID': '1723',
'units': '-500.0', 'price': 1.1911, 'guaranteedExecutionFee': '0.0',
'halfSpreadCost': '0.0441', 'initialMarginRequired': '16.65'},
'tradesClosed': [{'tradeID': '1721', 'units': '-1000.0', 'price':
1.1911, 'realizedPL': '-0.1772', 'finan

In [24]:
api.create_order(instrument, 500)   # ③ 买入 500，平掉空头

{'id': '1725', 'time': '2020-08-19T14:39:17.348231507Z', 'userID':
13834683, 'accountID': '101-004-13834683-001', 'batchID': '1724',
'requestID': '24716258589171775', 'type': 'ORDER_FILL', 'orderID':
'1724', 'instrument': 'EUR_USD', 'units': '500.0',
'gainQuoteHomeConversionFactor': '0.835313189428',
'lossQuoteHomeConversionFactor': '0.84370829686', 'price': 1.1913,
'fullVWAP': 1.1913, 'fullPrice': {'type': 'PRICE', 'bids': [{'price':
1.19104, 'liquidity': '9998500'}], 'asks': [{'price': 1.1913,
'liquidity': '9999000'}], 'closeoutBid': 1.19104, 'closeoutAsk':
1.1913}, 'reason': 'MARKET_ORDER', 'pl': '-0.0844', 'financing':
'0.0', 'commission': '0.0', 'guaranteedExecutionFee': '0.0',
'accountBalance': '98510.537', 'tradesClosed': [{'tradeID': '1723',
'units': '500.0', 'price': 1.1913, 'realizedPL': '-0.0844',
'financing': '0.0', 'guaranteedExecutionFee': '0.0', 'halfSpreadCost':
'0.0546'}], 'halfSpreadCost': '0.0546'}

虽然 Oanda API 支持多种订单类型，但本章及下一章主要使用市价单：每当出现新信号，就立即做多或做空。


## 6. Implementing Trading Strategies in Real Time（实时实现交易策略）

本节给出一个自定义类 `MomentumTrader`，它基于动量策略在 Oanda 上自动交易 `EUR_USD`。完整版见本 notebook 末尾「Python Script」一节。下面逐行讲解，先从 `__init__` 方法开始。该类继承自 `tpqoa` 类。

`__init__` 参数/属性说明：

| 属性 | 含义 |
|---|---|
| `self.position = 0` | 初始仓位（市场中性） |
| `self.instrument` | 要交易的品种 |
| `self.bar_length` | 对 tick 数据重采样的 K 线长度 |
| `self.momentum` | 计算动量用的区间个数 |
| `self.units` | 交易单位数量 |
| `self.raw_data = pd.DataFrame()` | 用来存放 tick 数据的空 DataFrame |
| `self.min_length = self.momentum + 1` | 开始交易所需的初始最小 K 线条数 |

核心方法是 `.on_success()`，它实现动量策略的交易逻辑，每当有新的 tick 数据到达时被调用。

下面的单元格把类写入文件 `MomentumTrader.py`（之后用 `import MomentumTrader as MT` 导入）：


In [25]:
%%writefile MomentumTrader.py
import tpqoa
import numpy as np
import pandas as pd

class MomentumTrader(tpqoa.tpqoa):
    def __init__(self, conf_file, instrument, bar_length, momentum, units,
                 *args, **kwargs):
        super(MomentumTrader, self).__init__(conf_file)   # ① 配置文件（含凭证）
        self.position = 0                  # ② 初始仓位值（市场中性）
        self.instrument = instrument       # ③ 要交易的品种
        self.momentum = momentum           # ④ 动量计算所用的区间个数
        self.bar_length = bar_length       # ⑤ 对 tick 数据重采样的 K 线长度
        self.units = units                 # ⑥ 交易单位数
        self.raw_data = pd.DataFrame()     # ⑦ 用于存放 tick 数据的空 DataFrame
        self.min_length = self.momentum + 1   # ⑧ 开始交易所需的初始最小 K 线条数

    def on_success(self, time, bid, ask):
        """ Takes actions when new tick data arrives. """   # ① 新 tick 到达时被调用
        print(self.ticks, end=' ')   # ② 打印已收到的 tick 数
        self.raw_data = self.raw_data.append(pd.DataFrame(
            {'bid': bid, 'ask': ask}, index=[pd.Timestamp(time)]))   # ③ 收集并存储 tick 数据
        self.data = self.raw_data.resample(
            self.bar_length, label='right').last().ffill().iloc[:-1]   # ④ 将 tick 重采样为指定长度的 K 线
        self.data['mid'] = self.data.mean(axis=1)   # ⑤ 计算中间价…
        self.data['returns'] = np.log(self.data['mid'] /
                                      self.data['mid'].shift(1))   # ⑥ …并据此计算对数收益率
        self.data['position'] = np.sign(
            self.data['returns'].rolling(self.momentum).mean())   # ⑦ 基于动量参数得出信号/仓位（在线算法）
        if len(self.data) > self.min_length:   # ⑧ 数据足够或有新数据时，执行交易逻辑，并每次把最小长度加 1
            self.min_length += 1
            if self.data['position'].iloc[-1] == 1:   # ⑨ 最新信号为 1（做多）？
                if self.position == 0:                # ⑩ 当前仓位为 0（中性）…
                    self.create_order(self.instrument, self.units)       # ⑪ …买入 self.units
                elif self.position == -1:             # ⑫ 当前为 -1（空头）…
                    self.create_order(self.instrument, self.units * 2)   # ⑬ …买入 2×units（先平空再开多；原书写作"买入 0"）
                self.position = 1                     # ⑭ 把仓位设为 +1（多头）
            elif self.data['position'].iloc[-1] == -1:   # ⑮ 最新信号为 -1（做空）？
                if self.position == 0:                # ⑯ 当前仓位为 0（中性）…
                    self.create_order(self.instrument, -self.units)      # ⑰ …卖出 -self.units
                elif self.position == 1:              # ⑱ 当前为 +1（多头）…
                    self.create_order(self.instrument, -self.units * 2)  # ⑲ …卖出 2×units（先平多再开空）
                self.position = -1                    # ⑳ 把仓位设为 -1（空头）

Writing MomentumTrader.py


基于这个类，开启一次自动化算法交易只需四行代码。


In [26]:
import MomentumTrader as MT

In [27]:
mt = MT.MomentumTrader('oanda.cfg',   # ① 含凭证的配置文件
                       instrument=instrument,   # ② 指定交易品种
                       bar_length='10s',   # ③ 重采样的 K 线长度
                       momentum=6,   # ④ 动量参数（作用于重采样后的区间）
                       units=10000)   # ⑤ 多/空仓位的交易单位数


下面启动数据流，也就启动了交易；收到 500 个 tick 后停止。

输出说明：数字是 `print(self.ticks)` 打印的 tick 序号；夹在数字之间的字典是成交回报（三次：开多 10000 → 反手为空 −20000 → 反手为多 +20000）。


In [28]:
mt.stream_data(mt.instrument, stop=500)   # ⑥ 开始流式数据并实时交易

1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27
28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50
51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73
74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96
97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114
115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131
132 133 134 135 136 137 138 139 140 141 142 143 144 145 146 147 148
149 150 151 152 153
{'id': '1727', 'time': '2020-08-19T14:40:30.443867492Z', 'userID':
13834683, 'accountID': '101-004-13834683-001', 'batchID': '1726',
'requestID': '42730657405829101', 'type': 'ORDER_FILL', 'orderID':
'1726', 'instrument': 'EUR_USD', 'units': '10000.0',
'gainQuoteHomeConversionFactor': '0.8350012403',
'lossQuoteHomeConversionFactor': '0.843393212565', 'price': 1.19168,
'fullVWAP': 1.19168, 'fullPrice': {'type': 'PRICE', 'bids': [{'price':
1.19155, 'liquidity': '10000000'}], 'asks': [{'price':

最后，平掉最终持仓：


In [29]:
oo = mt.create_order(instrument, units=-mt.position * mt.units,
                     ret=True, suppress=True)   # ① 平掉最后的仓位
oo

{'id': '1733',
 'time': '2020-08-19T14:43:17.107985242Z',
 'userID': 13834683,
 'accountID': '101-004-13834683-001',
 'batchID': '1732',
 'requestID': '42730658106750652',
 'type': 'ORDER_FILL',
 'orderID': '1732',
 'instrument': 'EUR_USD',
 'units': '-10000.0',
 'gainQuoteHomeConversionFactor': '0.835327206922',
 'lossQuoteHomeConversionFactor': '0.843722455232',
 'price': 1.19109,
 'fullVWAP': 1.19109,
 'fullPrice': {'type': 'PRICE',
  'bids': [{'price': 1.19109, 'liquidity': '10000000'}],
  'asks': [{'price': 1.19121, 'liquidity': '10000000'}],
  'closeoutBid': 1.19109,
  'closeoutAsk': 1.19121},
 'reason': 'MARKET_ORDER',
 'pl': '-0.1687',
 'financing': '0.0',
 'commission': '0.0',
 'guaranteedExecutionFee': '0.0',
 'accountBalance': '98507.7425',
 'tradesClosed': [{'tradeID': '1731',
   'units': '-10000.0',
   'price': 1.19109,
   'realizedPL': '-0.1687',
   'financing': '0.0',
   'guaranteedExecutionFee': '0.0',
   'halfSpreadCost': '0.5037'}],
 'halfSpreadCost': '0.5037'}

## 7. Retrieving Account Information（获取账户信息）

账户信息、交易历史等同样可通过 Oanda 的 RESTful API 方便地获取。例如，执行完上面的动量策略后，交易者可能想查看账户当前余额，用 `.get_account_summary()`：

In [30]:
api.get_account_summary()

{'id': '101-004-13834683-001',
 'alias': 'Primary',
 'currency': 'EUR',
 'balance': '98507.7425',
 'createdByUserID': 13834683,
 'createdTime': '2020-03-19T06:08:14.363139403Z',
 'guaranteedStopLossOrderMode': 'DISABLED',
 'pl': '-1273.126',
 'resettablePL': '-1273.126',
 'resettablePLTime': '0',
 'financing': '-219.1315',
 'commission': '0.0',
 'guaranteedExecutionFees': '0.0',
 'marginRate': '0.0333',
 'openTradeCount': 1,
 'openPositionCount': 1,
 'pendingOrderCount': 0,
 'hedgingEnabled': False,
 'unrealizedPL': '929.8862',
 'NAV': '99437.6287',
 'marginUsed': '377.76',
 'marginAvailable': '99064.4945',
 'positionValue': '3777.6',
 'marginCloseoutUnrealizedPL': '935.8183',
 'marginCloseoutNAV': '99443.5608',
 'marginCloseoutMarginUsed': '377.76',
 'marginCloseoutPercent': '0.0019',
 'marginCloseoutPositionValue': '3777.6',
 'withdrawalLimit': '98507.7425',
 'marginCallMarginUsed': '377.76',
 'marginCallPercent': '0.0038',
 'lastTransactionID': '1733'}

最近几笔交易的信息用 `.get_transactions()` 获取：

In [31]:
api.get_transactions(tid=int(oo['id']) - 2)

[{'id': '1732',
  'time': '2020-08-19T14:43:17.107985242Z',
  'userID': 13834683,
  'accountID': '101-004-13834683-001',
  'batchID': '1732',
  'requestID': '42730658106750652',
  'type': 'MARKET_ORDER',
  'instrument': 'EUR_USD',
  'units': '-10000.0',
  'timeInForce': 'FOK',
  'positionFill': 'DEFAULT',
  'reason': 'CLIENT_ORDER'},
 {'id': '1733',
  'time': '2020-08-19T14:43:17.107985242Z',
  'userID': 13834683,
  'accountID': '101-004-13834683-001',
  'batchID': '1732',
  'requestID': '42730658106750652',
  'type': 'ORDER_FILL',
  'orderID': '1732',
  'instrument': 'EUR_USD',
  'units': '-10000.0',
  'gainQuoteHomeConversionFactor': '0.835327206922',
  'lossQuoteHomeConversionFactor': '0.843722455232',
  'price': 1.19109,
  'fullVWAP': 1.19109,
  'fullPrice': {'type': 'PRICE',
   'bids': [{'price': 1.19109, 'liquidity': '10000000'}],
   'asks': [{'price': 1.19121, 'liquidity': '10000000'}],
   'closeoutBid': 1.19109,
   'closeoutAsk': 1.19121},
  'reason': 'MARKET_ORDER',
  'pl': '-

想要简明概览，可用 `.print_transactions()` 方法（列依次为：交易 ID ｜ 时间 ｜ 品种 ｜ 单位数 ｜ 已实现盈亏）：

In [32]:
api.print_transactions(tid=int(oo['id']) - 18)

1717 | 2020-08-19T14:37:00.803426931Z | EUR_USD | -10000.0 | 0.0
1719 | 2020-08-19T14:38:21.953399006Z | EUR_USD | 10000.0 | 6.8444
1721 | 2020-08-19T14:39:17.062399275Z | EUR_USD | 1000.0 | 0.0
1723 | 2020-08-19T14:39:17.200434462Z | EUR_USD | -1500.0 | -0.1772
1725 | 2020-08-19T14:39:17.348231507Z | EUR_USD | 500.0 | -0.0844
1727 | 2020-08-19T14:40:30.443867492Z | EUR_USD | 10000.0 | 0.0
1729 | 2020-08-19T14:41:11.436438078Z | EUR_USD | -20000.0 | -3.7118
1731 | 2020-08-19T14:42:20.525804142Z | EUR_USD | 20000.0 | 1.086
1733 | 2020-08-19T14:43:17.107985242Z | EUR_USD | -10000.0 | -0.1687


## 8. Conclusions（结论）

Oanda 平台让人可以轻松、直接地进入自动化算法交易的世界。Oanda 专注于差价合约（CFD）；视交易者所在国家不同，可交易的品种种类相当丰富。

从技术角度看，Oanda 的主要优势是现代、强大的 API，可通过专用的 Python 封装包（`v20`）轻松访问。

本章展示了：如何开户、如何用 Python 连接 API、如何获取（分钟线）历史数据用于回测、如何获取实时流式数据、如何基于动量策略自动交易 CFD，以及如何获取账户信息和详细的交易历史。

### 8.1 参考与延伸资源

- Oanda 的 *Help and Support* 页面：了解平台及 CFD 交易的重要方面
- Oanda 开发者门户 *Getting Started*：API 的详细说明


## 9. 附：Python Script

下面是包含 Oanda 自定义流式交易类、可自动交易动量策略的完整脚本。与第 6 节相比，多了文件头注释和 `if __name__ == '__main__':` 部分，里面预设了两套策略参数（`strat = 1` 用 `DE30_EUR`，`strat = 2` 用 `EUR_USD`）。

In [33]:
%%writefile momentum_trader_script.py
#
# Python Script
# with Momentum Trading Class
# for Oanda v20
#
# Python for Algorithmic Trading
# (c) Dr. Yves J. Hilpisch
# The Python Quants GmbH
#
import tpqoa
import numpy as np
import pandas as pd

class MomentumTrader(tpqoa.tpqoa):
    def __init__(self, conf_file, instrument, bar_length, momentum, units,
                 *args, **kwargs):
        super(MomentumTrader, self).__init__(conf_file)
        self.position = 0
        self.instrument = instrument
        self.momentum = momentum
        self.bar_length = bar_length
        self.units = units
        self.raw_data = pd.DataFrame()
        self.min_length = self.momentum + 1

    def on_success(self, time, bid, ask):
        """ Takes actions when new tick data arrives. """
        print(self.ticks, end=' ')
        self.raw_data = self.raw_data.append(pd.DataFrame(
            {'bid': bid, 'ask': ask}, index=[pd.Timestamp(time)]))
        self.data = self.raw_data.resample(
            self.bar_length, label='right').last().ffill().iloc[:-1]
        self.data['mid'] = self.data.mean(axis=1)
        self.data['returns'] = np.log(self.data['mid'] /
                                      self.data['mid'].shift(1))
        self.data['position'] = np.sign(
            self.data['returns'].rolling(self.momentum).mean())
        if len(self.data) > self.min_length:
            self.min_length += 1
            if self.data['position'].iloc[-1] == 1:
                if self.position == 0:
                    self.create_order(self.instrument, self.units)
                elif self.position == -1:
                    self.create_order(self.instrument, self.units * 2)
                self.position = 1
            elif self.data['position'].iloc[-1] == -1:
                if self.position == 0:
                    self.create_order(self.instrument, -self.units)
                elif self.position == 1:
                    self.create_order(self.instrument, -self.units * 2)
                self.position = -1

if __name__ == '__main__':
    strat = 2
    if strat == 1:
        mom = MomentumTrader('oanda.cfg', 'DE30_EUR', '5s', 3, 1)
        mom.stream_data(mom.instrument, stop=100)
        mom.create_order(mom.instrument, units=-mom.position * mom.units)
    elif strat == 2:
        mom = MomentumTrader('oanda.cfg', instrument='EUR_USD',
                             bar_length='5s', momentum=6, units=100000)
        mom.stream_data(mom.instrument, stop=100)
        mom.create_order(mom.instrument, units=-mom.position * mom.units)
    else:
        print('Strategy not known.')


Writing momentum_trader_script.py
